# 02 — End-to-End: weights → ONNX → CPU runtime benchmarks

Generic runner (works in Colab **or** locally). Loads trained weights, exports ONNX,
quantizes INT8, benchmarks CPU runtimes (ONNX Runtime / OpenVINO / NCNN), and prints
the final latency-vs-precision table for the README.


## 1. Setup

In [ ]:
REPO_URL = 'https://github.com/avneeshjadhav04/edge-vision-model'
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip -q install onnx onnxruntime onnxsim onnxscript onnxconverter-common openvino
    %cd /content
    !rm -rf edge-vision-model
    !git clone $REPO_URL
    %cd edge-vision-model
    sys.path.insert(0, '/content/edge-vision-model')


In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)


## 2. Get weights

Point `WEIGHTS` at your best checkpoint (Drive path in Colab, local path elsewhere).
`NUM_CLASSES`: 20 for VOC weights, 80 for COCO weights.


In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
WEIGHTS = 'runs/voc/best.pt'        # <-- EDIT
NUM_CLASSES = 20                     # <-- EDIT (20 voc / 80 coco)
IMG_SIZE = 640


## 3. Export ONNX (aux head stripped) + INT8

In [ ]:
import os
os.makedirs('runs/export', exist_ok=True)
from export.onnx_export import export_onnx
onnx_path = export_onnx(WEIGHTS, 'runs/export/evm_nano.onnx', NUM_CLASSES, IMG_SIZE)


In [ ]:
from export.quantize import quantize_int8
quantize_int8(onnx_path, 'runs/export/evm_nano_int8.onnx', IMG_SIZE)


## 4. Parity check: torch vs ORT on one image

In [ ]:
import numpy as np
import onnxruntime as ort
from models import build_model
from export.decode_onnx import decode_outputs
from scripts.common import load_config
cfg = load_config('model_nano')
m = build_model(cfg, num_classes=NUM_CLASSES)
import torch as _t
sd = _t.load(WEIGHTS, map_location='cpu', weights_only=False)
m.load_state_dict(sd.get('model', sd), strict=True); m.eval()
x = _t.randn(1, 3, IMG_SIZE, IMG_SIZE)
with _t.no_grad():
    r_t = m.predict(x, score_thresh=0.0, max_det=1000)
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
raw = sess.run(None, {'images': x.numpy()})
r_o = decode_outputs(raw, IMG_SIZE, num_classes=NUM_CLASSES, score_thresh=0.0, max_det=1000)[0]
st = r_t[0]['scores'].numpy(); so = r_o['scores']
print('max score torch/ort:', float(st.max()), float(so.max()))
assert abs(st.max() - so.max()) < 5e-3
print('PARITY OK')


## 5. CPU benchmarks (runtime x precision)

Median end-to-end latency over N runs on this machine's CPU (decode included).


In [ ]:
!python -m benchmarks.bench_runtime --onnx runs/export/evm_nano.onnx \
    --img-size $IMG_SIZE --n-iter 50 --int8 --openvino


## 6. mAP of the exported model (honest table: accuracy vs precision)

In [ ]:
# VOC example:
# !python -m scripts.eval --dataset voc --root ./datasets/VOC --weights $WEIGHTS --img-size $IMG_SIZE
# COCO example:
# !python -m scripts.eval --dataset coco --root ./datasets/coco --weights $WEIGHTS --img-size $IMG_SIZE
# INT8 mAP: run scripts/eval with an ORT-backed forward (see export/decode_onnx.decode_outputs
# wrapped over the val loader) - notebook cell below does VOC quickly:


In [ ]:
from data.voc import VOCDataset
from data.augment import EvalTransform
from data.common import collate_batch
from engine.eval_voc import eval_voc
from export.decode_onnx import decode_outputs
import torch as _t
ds = VOCDataset('./datasets/VOC', years=('2007',), split='test')
sub = _t.utils.data.Subset(ds, range(0, len(ds), 20))  # 5% subsample for a quick INT8 mAP
ds.transform = EvalTransform(IMG_SIZE)
dl = _t.utils.data.DataLoader(sub, batch_size=1, num_workers=2, collate_fn=collate_batch)
sess = ort.InferenceSession('runs/export/evm_nano_int8.onnx', providers=['CPUExecutionProvider'])
preds, tgts = [], []
from data.voc import VOC_CLASSES
for imgs, targets in dl:
    arr = (imgs[0].numpy().transpose(1, 2, 0) * 255).astype(np.uint8)
    from export.decode_onnx import preprocess as pp
    x, r, pads, orig = pp(arr, IMG_SIZE)
    raw = sess.run(None, {'images': x})
    d = decode_outputs(raw, IMG_SIZE, num_classes=NUM_CLASSES, score_thresh=0.01)[0]
    bb = d['pred_boxes']
    if bb.size:
        bb[:, [0, 2]] = (bb[:, [0, 2]] - pads[0]) / r
        bb[:, [1, 3]] = (bb[:, [1, 3]] - pads[1]) / r
    preds.append({'pred_boxes': _t.from_numpy(bb), 'scores': _t.from_numpy(d['scores']),
                  'labels': _t.from_numpy(d['labels'])})
    tgts.append(targets[0])
res = eval_voc(preds, tgts, num_classes=NUM_CLASSES if NUM_CLASSES == 20 else 20)
print('INT8 subset mAP@0.5 (5% of VOC test):', res['mAP'])


## 7. Latency-vs-mAP table (paste into README)

| Runtime | Precision | mean ms | p95 ms | FPS | mAP |
|---|---|---|---|---|---|
| ORT | FP32 | _fill_ | _fill_ | _fill_ | _fill_ |
| ORT | FP16 | _fill_ | _fill_ | _fill_ | _fill_ |
| ORT | INT8 | _fill_ | _fill_ | _fill_ | _fill_ |
| OpenVINO | FP32 | _fill_ | _fill_ | _fill_ | _fill_ |
| NCNN (ARM) | FP16 | _fill_ | _fill_ | _fill_ | _fill_ |


## 8. Optional: live webcam demo (local machine with webcam)

In [ ]:
# !python -m benchmarks.webcam_demo --onnx runs/export/evm_nano.onnx --num-classes 20 --names "aeroplane,bicycle,bird,boat,bottle,bus,car,cat,chair,cow,diningtable,dog,horse,motorbike,person,pottedplant,sheep,sofa,train,tvmonitor"
